# Segmentation & Classification Pipeline

Pipeline untuk:
1. **Image Enhancement** - CLAHE untuk low quality images
2. **Object Segmentation** - GrabCut + Morphological Operations untuk clothing
3. **Background Removal** - Masking object dan remove background
4. **Classification** - Ready untuk model training

**Note:** Optimized untuk objek clothing dengan warna serupa background.

## 1. Install Dependencies

In [ ]:
!pip install opencv-python pillow scikit-image scikit-learn matplotlib pandas numpy -q

print("✓ Dependencies installed")

## 2. Import Libraries

In [ ]:
import cv2
import numpy as np
from PIL import Image, ImageEnhance, ImageFilter
import matplotlib.pyplot as plt
import os
import pandas as pd
from skimage import morphology, filters, segmentation
from skimage.restoration import denoise_bilateral
from skimage.color import rgb2gray
from skimage.feature import canny
from scipy import ndimage

print("✓ Libraries imported")

## 3. Load Sample Images

In [ ]:
train_dir = 'train/train/'
train_df = pd.read_csv('train.csv')

def load_image(image_id, image_dir):
    """Load image dengan berbagai format"""
    for ext in ['.jpg', '.png', '.jpeg']:
        image_path = os.path.join(image_dir, f"{image_id}{ext}")
        if os.path.exists(image_path):
            return Image.open(image_path).convert('RGB')
    return None

# Load sample images
sample_ids = [425, 422, 421]
sample_images = []

for img_id in sample_ids:
    img = load_image(img_id, train_dir)
    if img is not None:
        sample_images.append((img_id, img))

print(f"✓ Loaded {len(sample_images)} sample images")
print(f"Sample IDs: {[img_id for img_id, _ in sample_images]}")

## 4. Image Enhancement dengan CLAHE

Preprocessing untuk low quality images tanpa merusak object boundaries.

In [ ]:
def enhance_image_clahe(image):
    """
    CLAHE-based enhancement untuk preserve object boundaries
    """
    img_array = np.array(image)
    
    # Mild denoising
    denoised = denoise_bilateral(img_array, sigma_color=0.03, sigma_spatial=10,
                                  channel_axis=-1)
    denoised = (denoised * 255).astype(np.uint8)
    
    # CLAHE pada L channel (LAB color space)
    img_lab = cv2.cvtColor(denoised, cv2.COLOR_RGB2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img_lab[:, :, 0] = clahe.apply(img_lab[:, :, 0])
    img_enhanced = cv2.cvtColor(img_lab, cv2.COLOR_LAB2RGB)
    
    img_pil = Image.fromarray(img_enhanced)
    
    # Gentle sharpening
    enhancer = ImageEnhance.Sharpness(img_pil)
    img_pil = enhancer.enhance(1.2)
    
    return img_pil

print("✓ Enhancement function defined")

## 5. Advanced Segmentation Functions

Kombinasi GrabCut + Morphological Operations + Edge Detection untuk segmentasi robust.

### Metode Segmentation Baru: Saliency + Watershed

**Mengapa GrabCut Buruk untuk Clothing:**
- ❌ Membutuhkan rectangle initialization yang akurat
- ❌ Gagal ketika object memiliki warna serupa background
- ❌ Tidak adaptif terhadap berbagai pose/bentuk clothing
- ❌ Iterasi tinggi tidak menjamin hasil baik

**Metode Baru: Saliency-based Segmentation**
- ✅ **Saliency Detection**: Otomatis fokus pada objek yang menonjol
- ✅ **Spectral Residual**: Deteksi objek tanpa prior knowledge
- ✅ **Otsu Thresholding**: Adaptive threshold untuk berbagai kondisi
- ✅ **Distance Transform + Watershed**: Precise object boundaries
- ✅ **Edge-aware**: Preserve boundaries dengan Canny edges
- ✅ **Morphological Refinement**: Smooth dan clean mask

**Pipeline 7 Langkah:**
1. Saliency detection (Spectral Residual)
2. Otsu thresholding (adaptive)
3. Morphological operations (open + close)
4. Distance transform + Watershed segmentation
5. Edge detection & integration
6. Largest component selection
7. Final smoothing & refinement

In [ ]:
def segment_clothing_advanced(image):
    """
    Advanced segmentation menggunakan Saliency + Watershed + Superpixel:
    1. Saliency detection untuk fokus pada objek utama
    2. Otsu thresholding untuk initial mask
    3. Distance transform + Watershed untuk precise boundaries
    4. Superpixel-based refinement
    5. Morphological operations untuk smooth mask
    
    Lebih robust untuk clothing/person dengan warna serupa background
    """
    img_array = np.array(image)
    height, width = img_array.shape[:2]
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    
    # ============================================================
    # STEP 1: Saliency Detection - Detect salient regions (object)
    # ============================================================
    # Spectral Residual Saliency
    saliency = cv2.saliency.StaticSaliencySpectralResidual_create()
    success, saliency_map = saliency.computeSaliency(img_array)
    saliency_map = (saliency_map * 255).astype(np.uint8)
    
    # Enhance saliency map
    saliency_enhanced = cv2.equalizeHist(saliency_map)
    
    # Apply CLAHE untuk better contrast
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    saliency_clahe = clahe.apply(saliency_enhanced)
    
    # ============================================================
    # STEP 2: Otsu Thresholding untuk initial mask
    # ============================================================
    # Gaussian blur untuk reduce noise
    blurred = cv2.GaussianBlur(saliency_clahe, (5, 5), 0)
    
    # Otsu's thresholding
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # ============================================================
    # STEP 3: Morphological Operations - Clean up mask
    # ============================================================
    # Remove noise
    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask_opened = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel_open, iterations=2)
    
    # Fill holes
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (21, 21))
    mask_closed = cv2.morphologyEx(mask_opened, cv2.MORPH_CLOSE, kernel_close, iterations=3)
    
    # ============================================================
    # STEP 4: Distance Transform + Watershed
    # ============================================================
    # Distance transform
    dist_transform = cv2.distanceTransform(mask_closed, cv2.DIST_L2, 5)
    
    # Threshold untuk get sure foreground
    _, sure_fg = cv2.threshold(dist_transform, 0.3 * dist_transform.max(), 255, 0)
    sure_fg = np.uint8(sure_fg)
    
    # Dilate untuk get sure background
    kernel_dilate = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    sure_bg = cv2.dilate(mask_closed, kernel_dilate, iterations=3)
    
    # Find unknown region
    unknown = cv2.subtract(sure_bg, sure_fg)
    
    # Label markers untuk watershed
    _, markers = cv2.connectedComponents(sure_fg)
    markers = markers + 1
    markers[unknown == 255] = 0
    
    # Apply watershed
    img_array_copy = img_array.copy()
    markers = cv2.watershed(img_array_copy, markers)
    
    # Create mask from watershed result (exclude boundaries marked as -1)
    mask_watershed = np.zeros_like(gray, dtype=np.uint8)
    mask_watershed[markers > 1] = 255
    
    # ============================================================
    # STEP 5: Edge-aware refinement
    # ============================================================
    # Detect edges untuk preserve boundaries
    edges = cv2.Canny(gray, 50, 150)
    
    # Dilate edges slightly
    kernel_edge = np.ones((2, 2), np.uint8)
    edges_dilated = cv2.dilate(edges, kernel_edge, iterations=1)
    
    # Combine dengan mask (add edges to object regions)
    mask_with_edges = cv2.bitwise_or(mask_watershed, edges_dilated)
    
    # Close small gaps
    kernel_final = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    mask_refined = cv2.morphologyEx(mask_with_edges, cv2.MORPH_CLOSE, kernel_final)
    
    # ============================================================
    # STEP 6: Keep largest connected component
    # ============================================================
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        mask_refined, connectivity=8)
    
    if num_labels > 1:
        # Get largest component
        largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        mask_largest = (labels == largest_label).astype('uint8') * 255
    else:
        mask_largest = mask_refined
    
    # ============================================================
    # STEP 7: Final smoothing
    # ============================================================
    # Smooth boundaries
    kernel_smooth = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask_smooth = cv2.morphologyEx(mask_largest, cv2.MORPH_CLOSE, kernel_smooth)
    mask_smooth = cv2.morphologyEx(mask_smooth, cv2.MORPH_OPEN, kernel_smooth)
    
    # Gaussian blur untuk very smooth edges
    mask_final = cv2.GaussianBlur(mask_smooth, (5, 5), 0)
    _, mask_final = cv2.threshold(mask_final, 127, 255, cv2.THRESH_BINARY)
    
    return mask_final.astype('uint8')

def apply_mask_and_remove_bg(image, mask, bg_color=(255, 255, 255)):
    """
    Apply mask dan remove background
    
    Args:
        image: PIL Image
        mask: numpy array (binary mask)
        bg_color: tuple untuk background color (default: white)
    
    Returns:
        result_image: PIL Image dengan background removed
    """
    img_array = np.array(image)
    
    # Ensure mask is binary
    mask_binary = (mask > 0).astype(np.uint8)
    
    # Create background
    background = np.full_like(img_array, bg_color, dtype=np.uint8)
    
    # Apply mask
    result = np.where(mask_binary[:, :, np.newaxis] == 1, img_array, background)
    
    return Image.fromarray(result.astype(np.uint8))

print("✓ Segmentation functions defined")

## 6. Cropping & Resizing Functions

In [ ]:
def get_mask_bounding_box(mask):
    """Get bounding box dari mask"""
    rows = np.any(mask > 0, axis=1)
    cols = np.any(mask > 0, axis=0)
    
    if not np.any(rows) or not np.any(cols):
        return None
    
    y_min, y_max = np.where(rows)[0][[0, -1]]
    x_min, x_max = np.where(cols)[0][[0, -1]]
    
    return (x_min, y_min, x_max, y_max)

def crop_with_padding(image, bbox, padding_percent=0.05):
    """Crop image berdasarkan bbox dengan padding"""
    if bbox is None:
        return image, {'cropped': False}
    
    x_min, y_min, x_max, y_max = bbox
    width, height = image.size
    
    # Calculate padding
    obj_width = x_max - x_min
    obj_height = y_max - y_min
    pad_x = int(obj_width * padding_percent)
    pad_y = int(obj_height * padding_percent)
    
    # Apply padding with bounds
    x_min = max(0, x_min - pad_x)
    y_min = max(0, y_min - pad_y)
    x_max = min(width, x_max + pad_x)
    y_max = min(height, y_max + pad_y)
    
    cropped = image.crop((x_min, y_min, x_max, y_max))
    
    crop_info = {
        'cropped': True,
        'original_size': (width, height),
        'crop_box': (x_min, y_min, x_max, y_max),
        'cropped_size': cropped.size
    }
    
    return cropped, crop_info

def resize_to_target(image, target_size=(224, 224)):
    """Resize maintaining aspect ratio dengan white padding"""
    width, height = image.size
    target_w, target_h = target_size
    
    img_ratio = width / height
    target_ratio = target_w / target_h
    
    if img_ratio > target_ratio:
        new_width = target_w
        new_height = int(target_w / img_ratio)
    else:
        new_height = target_h
        new_width = int(target_h * img_ratio)
    
    resized = image.resize((new_width, new_height), Image.LANCZOS)
    
    # Pad dengan white background
    final_image = Image.new('RGB', target_size, (255, 255, 255))
    paste_x = (target_w - new_width) // 2
    paste_y = (target_h - new_height) // 2
    final_image.paste(resized, (paste_x, paste_y))
    
    return final_image

print("✓ Cropping & resizing functions defined")

## 7. Complete Pipeline

Pipeline lengkap: Enhancement → Segmentation → BG Removal → Crop → Resize

In [ ]:
def process_clothing_image(image, target_size=(224, 224)):
    """
    Complete pipeline untuk clothing classification:
    1. CLAHE enhancement (preserve boundaries)
    2. Advanced segmentation (GrabCut + Morphology + Edges)
    3. Remove background
    4. Crop to object
    5. Resize to target size
    
    Returns:
        dict dengan semua intermediate results
    """
    print("  Step 1: Enhancing with CLAHE...")
    enhanced = enhance_image_clahe(image)
    
    print("  Step 2: Segmenting clothing object...")
    mask = segment_clothing_advanced(enhanced)
    
    print("  Step 3: Removing background...")
    nobg = apply_mask_and_remove_bg(enhanced, mask)
    
    print("  Step 4: Getting bounding box...")
    bbox = get_mask_bounding_box(mask)
    
    if bbox:
        print(f"    ✓ Object found at: {bbox}")
    else:
        print("    ⚠ No object detected")
        bbox = (0, 0, image.size[0], image.size[1])
    
    print("  Step 5: Cropping with padding...")
    cropped, crop_info = crop_with_padding(nobg, bbox, padding_percent=0.05)
    
    if crop_info['cropped']:
        print(f"    ✓ Cropped from {crop_info['original_size']} to {crop_info['cropped_size']}")
    
    print(f"  Step 6: Resizing to {target_size}...")
    final = resize_to_target(cropped, target_size)
    
    return {
        'original': image,
        'enhanced': enhanced,
        'mask': mask,
        'nobg': nobg,
        'bbox': bbox,
        'cropped': cropped,
        'final': final,
        'crop_info': crop_info
    }

print("✓ Complete pipeline defined")

## 8. Process Sample Images

Proses semua sample images dengan pipeline lengkap.

In [ ]:
results = []

for img_id, img in sample_images:
    print(f"\n{'='*70}")
    print(f"Processing Image ID: {img_id}")
    print('='*70)
    
    result = process_clothing_image(img, target_size=(224, 224))
    result['id'] = img_id
    results.append(result)
    
    print(f"✓ Image {img_id} completed\n")

print(f"\n{'='*70}")
print(f"✓ Successfully processed {len(results)} images!")
print('='*70)

## 9. Visualisasi: Complete Pipeline

Tampilkan semua tahap processing (6 columns)

In [ ]:
fig, axes = plt.subplots(len(results), 6, figsize=(30, 5*len(results)))
if len(results) == 1:
    axes = [axes]

for idx, result in enumerate(results):
    img_id = result['id']
    
    # Column 1: Original
    axes[idx][0].imshow(result['original'])
    axes[idx][0].set_title(f'1. Original\nID: {img_id}', fontsize=10, fontweight='bold')
    axes[idx][0].axis('off')
    
    # Column 2: Enhanced (CLAHE)
    axes[idx][1].imshow(result['enhanced'])
    axes[idx][1].set_title('2. Enhanced\n(CLAHE)', fontsize=10, fontweight='bold', color='blue')
    axes[idx][1].axis('off')
    
    # Column 3: Segmentation Mask
    axes[idx][2].imshow(result['mask'], cmap='gray')
    axes[idx][2].set_title('3. Mask\n(Segmentation)', fontsize=10, fontweight='bold', color='purple')
    axes[idx][2].axis('off')
    
    # Column 4: Background Removed
    axes[idx][3].imshow(result['nobg'])
    bbox = result['bbox']
    rect = plt.Rectangle((bbox[0], bbox[1]), bbox[2]-bbox[0], bbox[3]-bbox[1],
                         fill=False, edgecolor='red', linewidth=3)
    axes[idx][3].add_patch(rect)
    axes[idx][3].set_title('4. BG Removed\n+ BBox', fontsize=10, fontweight='bold', color='green')
    axes[idx][3].axis('off')
    
    # Column 5: Cropped
    axes[idx][4].imshow(result['cropped'])
    axes[idx][4].set_title(f'5. Cropped\n{result["cropped"].size}', fontsize=10, fontweight='bold', color='orange')
    axes[idx][4].axis('off')
    
    # Column 6: Final
    axes[idx][5].imshow(result['final'])
    axes[idx][5].set_title(f'6. Final\n{result["final"].size}', fontsize=10, fontweight='bold', color='red')
    axes[idx][5].axis('off')

plt.tight_layout()
plt.savefig('segmentation_classification_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Visualization saved: segmentation_classification_pipeline.png")

## 10. Visualisasi: Segmentation Quality

Detail visualisasi dari kualitas segmentation mask.

In [ ]:
fig, axes = plt.subplots(len(results), 4, figsize=(20, 5*len(results)))
if len(results) == 1:
    axes = [axes]

for idx, result in enumerate(results):
    img_id = result['id']
    mask = result['mask']
    
    # Column 1: Original
    axes[idx][0].imshow(result['original'])
    axes[idx][0].set_title(f'Original\nID: {img_id}', fontsize=12, fontweight='bold')
    axes[idx][0].axis('off')
    
    # Column 2: Mask Overlay (Green)
    overlay = np.array(result['enhanced']).copy()
    green_mask = np.zeros_like(overlay)
    green_mask[:,:,1] = mask * 255  # Green channel
    overlay = cv2.addWeighted(overlay, 0.7, green_mask, 0.3, 0)
    axes[idx][1].imshow(overlay)
    axes[idx][1].set_title('Mask Overlay\n(Green = Object)', fontsize=12, fontweight='bold', color='green')
    axes[idx][1].axis('off')
    
    # Column 3: Mask + Contours
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    img_with_contours = np.array(result['enhanced']).copy()
    cv2.drawContours(img_with_contours, contours, -1, (255, 0, 0), 3)
    axes[idx][2].imshow(img_with_contours)
    axes[idx][2].set_title(f'Boundaries\n({len(contours)} contours)', fontsize=12, fontweight='bold', color='red')
    axes[idx][2].axis('off')
    
    # Column 4: Final Result
    axes[idx][3].imshow(result['nobg'])
    axes[idx][3].set_title('Final Segmented\n(BG Removed)', fontsize=12, fontweight='bold', color='blue')
    axes[idx][3].axis('off')

plt.tight_layout()
plt.savefig('segmentation_quality.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Segmentation quality visualization saved: segmentation_quality.png")

## 10.5. Visualisasi: Intermediate Segmentation Steps

Tampilkan setiap tahap dari segmentation pipeline untuk debugging.

In [ ]:
def visualize_segmentation_steps(image):
    """Visualize all intermediate steps of segmentation"""
    img_array = np.array(image)
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    
    # Step 1: Saliency
    saliency = cv2.saliency.StaticSaliencySpectralResidual_create()
    success, saliency_map = saliency.computeSaliency(img_array)
    saliency_map = (saliency_map * 255).astype(np.uint8)
    saliency_enhanced = cv2.equalizeHist(saliency_map)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    saliency_clahe = clahe.apply(saliency_enhanced)
    
    # Step 2: Otsu thresholding
    blurred = cv2.GaussianBlur(saliency_clahe, (5, 5), 0)
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # Step 3: Morphological operations
    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask_opened = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel_open, iterations=2)
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (21, 21))
    mask_closed = cv2.morphologyEx(mask_opened, cv2.MORPH_CLOSE, kernel_close, iterations=3)
    
    # Step 4: Distance transform
    dist_transform = cv2.distanceTransform(mask_closed, cv2.DIST_L2, 5)
    dist_visual = cv2.normalize(dist_transform, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    
    # Step 5: Final mask
    final_mask = segment_clothing_advanced(image)
    
    return {
        'saliency': saliency_clahe,
        'threshold': thresh,
        'morphology': mask_closed,
        'distance': dist_visual,
        'final': final_mask
    }

# Visualize untuk image pertama
if results:
    test_result = results[0]
    test_img = test_result['original']
    
    print("Generating intermediate steps visualization...")
    steps = visualize_segmentation_steps(test_img)
    
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    
    # Row 1
    axes[0][0].imshow(test_img)
    axes[0][0].set_title('Original Image', fontsize=12, fontweight='bold')
    axes[0][0].axis('off')
    
    axes[0][1].imshow(steps['saliency'], cmap='hot')
    axes[0][1].set_title('Step 1: Saliency Map\n(Spectral Residual)', fontsize=11, fontweight='bold')
    axes[0][1].axis('off')
    
    axes[0][2].imshow(steps['threshold'], cmap='gray')
    axes[0][2].set_title('Step 2: Otsu Threshold\n(Adaptive)', fontsize=11, fontweight='bold')
    axes[0][2].axis('off')
    
    axes[0][3].imshow(steps['morphology'], cmap='gray')
    axes[0][3].set_title('Step 3: Morphology\n(Open + Close)', fontsize=11, fontweight='bold')
    axes[0][3].axis('off')
    
    # Row 2
    axes[1][0].imshow(steps['distance'], cmap='jet')
    axes[1][0].set_title('Step 4: Distance Transform\n(For Watershed)', fontsize=11, fontweight='bold')
    axes[1][0].axis('off')
    
    axes[1][1].imshow(steps['final'], cmap='gray')
    axes[1][1].set_title('Step 5: Watershed Result\n(Precise Boundaries)', fontsize=11, fontweight='bold')
    axes[1][1].axis('off')
    
    # Apply mask
    img_masked = apply_mask_and_remove_bg(test_img, steps['final'])
    axes[1][2].imshow(img_masked)
    axes[1][2].set_title('Step 6: Background Removed\n(White BG)', fontsize=11, fontweight='bold')
    axes[1][2].axis('off')
    
    # Final with overlay
    overlay = np.array(test_img).copy()
    green_mask = np.zeros_like(overlay)
    green_mask[:,:,1] = steps['final']
    overlay = cv2.addWeighted(overlay, 0.6, green_mask, 0.4, 0)
    axes[1][3].imshow(overlay)
    axes[1][3].set_title('Step 7: Final Overlay\n(Green = Object)', fontsize=11, fontweight='bold', color='green')
    axes[1][3].axis('off')
    
    plt.tight_layout()
    plt.savefig('segmentation_steps_detailed.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("✓ Intermediate steps visualization saved: segmentation_steps_detailed.png")

## 11. Summary & Technical Details

Penjelasan teknis dari segmentation pipeline.

In [ ]:
print("="*80)
print("ADVANCED SEGMENTATION PIPELINE SUMMARY")
print("="*80)

print("\n📋 Pipeline Stages:")
print("-" * 80)
print("1. CLAHE Enhancement")
print("   • Adaptive contrast untuk preserve boundaries")
print("   • clipLimit=2.0, tileGridSize=(8,8)")
print("   • Tidak merusak object dengan warna serupa background")

print("\n2. Saliency Detection (Spectral Residual)")
print("   • Automatically detect salient regions")
print("   • Focus on main object (clothing/person)")
print("   • No rectangle initialization needed")
print("   • Adaptive to various object shapes")

print("\n3. Otsu Thresholding")
print("   • Automatic adaptive threshold")
print("   • Works with various lighting conditions")
print("   • Better than fixed threshold")

print("\n4. Morphological Operations")
print("   • Opening (5x5 ellipse, 2 iterations): Remove noise")
print("   • Closing (21x21 ellipse, 3 iterations): Fill holes")
print("   • Preserve object shape")

print("\n5. Distance Transform + Watershed")
print("   • Precise object boundary detection")
print("   • Separate overlapping regions")
print("   • Better than simple thresholding")

print("\n6. Edge-aware Refinement")
print("   • Canny edge detection (50-150)")
print("   • Integrate edges with mask")
print("   • Preserve fine boundaries")

print("\n7. Component Analysis & Smoothing")
print("   • Keep largest component only")
print("   • Gaussian blur for smooth edges")
print("   • Final morphological refinement")

print("\n8. Final Processing")
print("   • Background removal dengan white fill")
print("   • Crop to object + 5% padding")
print("   • Resize to 224x224 (aspect ratio maintained)")

print("\n" + "="*80)
print("✅ ADVANTAGES vs GrabCut & DeepLabV3:")
print("="*80)
print("• ✓ No manual rectangle initialization (vs GrabCut)")
print("• ✓ Better for clothing dengan warna serupa background")
print("• ✓ Saliency detection automatically finds object")
print("• ✓ Watershed provides precise boundaries")
print("• ✓ Works well dengan low quality images")
print("• ✓ No pretrained model needed (vs DeepLabV3)")
print("• ✓ Faster inference (no deep learning)")
print("• ✓ Object tidak kehapus seperti DeepLabV3")
print("• ✓ Adaptive threshold untuk berbagai kondisi")

print("\n" + "="*80)
print("📊 Processing Statistics:")
print("="*80)

for result in results:
    img_id = result['id']
    mask = result['mask']
    
    # Calculate statistics
    total_pixels = mask.size
    foreground_pixels = np.sum(mask > 0)
    background_pixels = total_pixels - foreground_pixels
    fg_percentage = (foreground_pixels / total_pixels) * 100
    
    print(f"\nImage ID: {img_id}")
    print(f"  • Foreground: {foreground_pixels:,} pixels ({fg_percentage:.1f}%)")
    print(f"  • Background: {background_pixels:,} pixels ({100-fg_percentage:.1f}%)")
    print(f"  • Original size: {result['crop_info']['original_size']}")
    print(f"  • Cropped size: {result['crop_info']['cropped_size']}")
    print(f"  • Final size: {result['final'].size}")

print("\n" + "="*80)